<a href="https://colab.research.google.com/github/sonyxoxo/ipd/blob/main/06_%D0%BF%D1%80%D0%BE%D0%B2%D0%B5%D1%80%D0%BA%D0%B0_%D0%BD%D0%B0%D1%83%D1%87%D0%BD%D1%8B%D1%85_%D0%B3%D0%B8%D0%BF%D0%BE%D1%82%D0%B5%D0%B7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Этап 6. Проверка научных гипотез

**Тема исследования:** «Изменение суточного профиля электропотребления домохозяйства: влияние полноты исходных данных и методов их обработки на результаты анализа».

Для каждого совместимого исходного файла независимо рассматриваются три гипотезы:

- **H1.** Суточные профили разных месяцев различаются по форме, амплитуде или времени пика.
- **H2.** Доля и распределение пропусков влияют на оценки суточного профиля.
- **H3.** Способ обработки неполных данных влияет на оценку профиля.

Исходные измерения не изменяются. Если для конкретного теста недостаточно данных, его результат не рассчитывается.

## 6.1. Исходные данные

Обрабатываются файлы форматов TXT, CSV, TSV и ZIP из `data/raw`. Необходимы поминутные временные метки и активная мощность с известной единицей измерения. При нестандартных названиях столбцов соответствие определяется в `analysis_config.json`. Результаты разных источников сохраняются раздельно.

In [1]:
from pathlib import Path
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

repo = Path("/content/ipd")
if (repo / ".git").is_dir():
    subprocess.run(["git", "-C", str(repo), "pull", "--ff-only"], check=True)
elif not repo.exists():
    subprocess.run(["git", "clone", "https://github.com/sonyxoxo/ipd.git", str(repo)], check=True)
else:
    raise RuntimeError("Папка /content/ipd существует, но не является клоном GitHub")
sys.path.insert(0, str(repo))

from ipd_analysis import (input_files, load_measurements, regular_minutes,
                          monthly_coverage, result_folder, save_csv, save_run_metadata)

sources = input_files()
print("Исходных файлов:", len(sources))
for file in sources:
    print(" •", file.relative_to(repo))

Исходных файлов: 1
 • data/raw/household_power_consumption.zip


## 6.2. H1 — сравнение суточных профилей

Единица анализа — сутки, а не минута. Соседние сутки также могут быть зависимы. Рассматриваются 24 почасовых средних значения за полные сутки. Проверяются форма профиля, суточный размах мощности и время максимума. Для каждой характеристики нулевая гипотеза предполагает равенство средних характеристик сравниваемых месяцев.

Статистика сравнения групп:

\[
T=\sum_m n_m\left\|\overline{X}_m-\overline{X}\right\|^2.
\]

Выполняются 999 перестановок блоков по 3 и 5 последовательных суток. Для трёх характеристик применяется поправка Холма. Доверительный интервал разности средней суточной мощности двух периодов оценивается бутстрэпом пятидневных блоков. Выводы зависят от допущения о сопоставимости блоков между периодами.

In [2]:
def block_groups(values, month_labels, selected_months, length):
    groups = []
    for month in selected_months:
        part = values[month_labels == month]
        size = len(part) // length
        if size == 0:
            return []
        blocks = part[:size * length].reshape(size, length, -1).mean(axis=1)
        groups.append(blocks[np.isfinite(blocks).all(axis=1)])
    return groups if all(len(g) >= 2 for g in groups) else []


def group_statistic(groups):
    total = np.concatenate(groups)
    center = total.mean(axis=0)
    return sum(len(g) * np.sum((g.mean(axis=0) - center) ** 2) for g in groups)


def permutation_p(groups, rng, repetitions=999):
    values = np.concatenate(groups)
    splits = np.cumsum([len(g) for g in groups])[:-1]
    observed = group_statistic(groups)
    exceeded = sum(group_statistic(np.split(rng.permutation(values), splits)) >= observed
                   for _ in range(repetitions))
    return (exceeded + 1) / (repetitions + 1)


def holm(pvalues):
    order = np.argsort(pvalues)
    adjusted = np.maximum.accumulate(np.asarray(pvalues)[order] *
                                     np.arange(len(order), 0, -1))
    result = np.empty(len(order))
    result[order] = np.minimum(adjusted, 1)
    return result


def analyze_h1(minutes, coverage, out):
    # Сначала используются ранее выбранные периоды, если они подходят источнику.
    eligible = coverage.loc[(coverage.expected_in_record == coverage.expected_calendar) &
                            (coverage.coverage_calendar_pct >= 99.9), "month"].tolist()
    preferred = ["2008-01", "2008-04", "2008-07", "2008-09"]
    selected = preferred if all(m in eligible for m in preferred) else eligible[:4]
    if len(selected) < 2:
        return "недостаточно полных месяцев"

    data = minutes.loc[minutes.timestamp.dt.strftime("%Y-%m").isin(selected),
                       ["timestamp", "Global_active_power"]].copy()
    data["month"] = data.timestamp.dt.strftime("%Y-%m")
    data["date"] = data.timestamp.dt.normalize()
    data["minute"] = data.timestamp.dt.hour * 60 + data.timestamp.dt.minute
    days = data.pivot(index=["month", "date"], columns="minute",
                      values="Global_active_power").reindex(columns=range(1440))
    month_labels = days.index.get_level_values("month").to_numpy()
    hourly = days.to_numpy().reshape(-1, 24, 60).mean(axis=2)
    mean_power = hourly.mean(axis=1)
    shape = np.divide(hourly, mean_power[:, None],
                      out=np.full_like(hourly, np.nan), where=mean_power[:, None] > 0)
    spread = (hourly.max(axis=1) - hourly.min(axis=1))[:, None]
    angle = 2 * np.pi * hourly.argmax(axis=1) / 24
    peak = np.column_stack([np.cos(angle), np.sin(angle)])
    peak[~np.isfinite(hourly).all(axis=1)] = np.nan
    characteristics = {"shape": shape, "amplitude": spread, "peak_hour": peak}

    results = []
    for length in (3, 5):
        rng = np.random.default_rng(2026 + length)
        names, pvalues = [], []
        for name, values in characteristics.items():
            groups = block_groups(values, month_labels, selected, length)
            if groups:
                names.append(name)
                pvalues.append(permutation_p(groups, rng))
        if pvalues:
            for name, p, adj in zip(names, pvalues, holm(pvalues)):
                results.append({"characteristic": name, "block_days": length,
                                "p_value": p, "p_holm": adj})
    if not results:
        return "недостаточно последовательных полных суток"
    save_csv(pd.DataFrame(results), out / "06_h1_tests.csv")

    contrast = ("2008-01", "2008-07") if {"2008-01", "2008-07"}.issubset(selected) else (selected[0], selected[-1])
    five_day = block_groups(mean_power[:, None], month_labels, selected, 5)
    if five_day:
        first = five_day[selected.index(contrast[0])].ravel()
        second = five_day[selected.index(contrast[1])].ravel()
        rng = np.random.default_rng(2036)
        draws = [rng.choice(first, len(first), replace=True).mean() -
                 rng.choice(second, len(second), replace=True).mean() for _ in range(999)]
        low, high = np.quantile(draws, [0.025, 0.975])
        save_csv(pd.DataFrame([{"first_month": contrast[0], "second_month": contrast[1],
                                "difference_kw": first.mean() - second.mean(),
                                "ci95_lower_kw": low, "ci95_upper_kw": high}]),
                 out / "06_h1_effect.csv")

    fig, ax = plt.subplots(figsize=(8, 4))
    for month in selected:
        group = shape[(month_labels == month) & np.isfinite(shape).all(axis=1)]
        if len(group):
            ax.plot(range(24), group.mean(axis=0), label=month)
    ax.set(xlabel="Час суток", ylabel="Относительная мощность",
           title="Нормированные суточные профили")
    ax.legend()
    ax.grid(alpha=0.2)
    fig.tight_layout()
    fig.savefig(out / "06_h1_profiles.png", dpi=160)
    plt.close(fig)
    return ", ".join(selected)

## 6.3. H2 — влияние пропусков

Контрольным периодом служит последовательность не менее 14 полностью измеренных суток. Для исходного набора UCI используется сентябрь 2008 года, если он доступен. Рассматриваются потери 1%, 5%, 10% и 20% измерений: случайные минуты, один непрерывный интервал и измерения только в вечерние часы (18:00–23:59). Для каждой комбинации выполняется 100 повторений с фиксированным начальным состоянием генератора.

Отклонение среднего суточного профиля от контрольного оценивается по MAE и RMSE (кВт), изменению максимума (%) и доле определённых минут профиля. Эти характеристики описывают влияние заданных механизмов пропусков, но не доказывают механизм естественных потерь.

In [3]:
def select_control(minutes):
    selected = minutes[["timestamp", "Global_active_power"]].copy()
    selected["date"] = selected.timestamp.dt.normalize()
    selected["minute"] = selected.timestamp.dt.hour * 60 + selected.timestamp.dt.minute
    daily = selected.pivot(index="date", columns="minute", values="Global_active_power")
    daily = daily.reindex(columns=range(1440))
    reference_month = daily.loc["2008-09-01":"2008-09-30"]
    if len(reference_month) == 30 and reference_month.notna().all().all():
        return reference_month

    complete = daily.notna().all(axis=1).to_numpy()
    starts = np.flatnonzero(complete & ~np.r_[False, complete[:-1]])
    ends = np.flatnonzero(complete & ~np.r_[complete[1:], False]) + 1
    if len(starts) == 0:
        return None
    position = np.argmax(ends - starts)
    start, end = starts[position], ends[position]
    if end - start < 14:
        return None
    return daily.iloc[start:min(end, start + 30)]


def missing_mask(n_days, fraction, scenario, rng):
    n = n_days * 1440
    count = round(n * fraction)
    mask = np.zeros(n, dtype=bool)
    if scenario == "continuous":
        start = rng.integers(0, n - count + 1)
        mask[start:start + count] = True
    else:
        candidates = (np.arange(n) if scenario == "random" else
                      np.flatnonzero(np.tile(np.arange(1440) // 60 >= 18, n_days)))
        mask[rng.choice(candidates, size=count, replace=False)] = True
    return mask


def profile_metrics(values, reference):
    array = values.reshape(-1, 1440)
    count = np.isfinite(array).sum(axis=0)
    profile = np.divide(np.nansum(array, axis=0), count,
                        out=np.full(1440, np.nan), where=count > 0)
    diff = profile - reference
    known = np.isfinite(diff)
    if not known.any():
        return {"mae_kw": np.nan, "rmse_kw": np.nan,
                "peak_change_pct": np.nan, "profile_coverage_pct": 0.0}
    return {"mae_kw": np.mean(np.abs(diff[known])),
            "rmse_kw": np.sqrt(np.mean(diff[known] ** 2)),
            "peak_change_pct": 100 * (np.nanmax(profile) / reference.max() - 1)
                               if reference.max() > 0 else np.nan,
            "profile_coverage_pct": 100 * known.mean()}


def analyze_h2(control, out):
    baseline = control.to_numpy()
    reference = baseline.mean(axis=0)
    values = baseline.ravel()
    rng = np.random.default_rng(2026)
    records = []
    for fraction in (0.01, 0.05, 0.10, 0.20):
        for scenario in ("random", "continuous", "evening"):
            for repetition in range(100):
                mask = missing_mask(len(control), fraction, scenario, rng)
                incomplete = values.copy()
                incomplete[mask] = np.nan
                records.append({"fraction": fraction, "scenario": scenario,
                                "repetition": repetition + 1,
                                **profile_metrics(incomplete, reference)})
    trials = pd.DataFrame(records)
    summary = trials.groupby(["fraction", "scenario"], as_index=False).agg(
        mean_mae_kw=("mae_kw", "mean"),
        p025_mae_kw=("mae_kw", lambda x: x.quantile(0.025)),
        p975_mae_kw=("mae_kw", lambda x: x.quantile(0.975)),
        mean_rmse_kw=("rmse_kw", "mean"),
        mean_peak_change_pct=("peak_change_pct", "mean"),
        mean_profile_coverage_pct=("profile_coverage_pct", "mean"))
    save_csv(trials, out / "06_h2_trials.csv")
    save_csv(summary, out / "06_h2_summary.csv")
    fig, ax = plt.subplots(figsize=(8, 4))
    for scenario, part in summary.groupby("scenario"):
        ax.plot(part.fraction * 100, part.mean_mae_kw, marker="o", label=scenario)
    ax.set(xlabel="Исключено измерений, %", ylabel="Средняя MAE, кВт",
           title="Чувствительность к пропускам")
    ax.grid(alpha=0.2)
    ax.legend()
    fig.tight_layout()
    fig.savefig(out / "06_h2_missing.png", dpi=160)
    plt.close(fig)
    return reference

## 6.4. H3 — сравнение методов обработки

При одних и тех же масках с 10% пропусков сравниваются усреднение по доступным наблюдениям, линейная интерполяция пропусков не длиннее 15 и 60 минут, а также усреднение только по полностью измеренным суткам. Для каждого механизма потерь используется 30 повторений. Критерий качества — MAE относительно известного контрольного профиля. Неприменимый метод отмечается отсутствующим результатом.

In [4]:
def interpolate_short(values, max_gap):
    series = pd.Series(values)
    missing = series.isna().to_numpy()
    interpolated = series.interpolate(limit_area="inside").to_numpy()
    starts = np.flatnonzero(missing & ~np.r_[False, missing[:-1]])
    ends = np.flatnonzero(missing & ~np.r_[missing[1:], False]) + 1
    for start, end in zip(starts, ends):
        if end - start > max_gap:
            interpolated[start:end] = np.nan
    return interpolated


def analyze_h3(control, reference, out):
    baseline = control.to_numpy().ravel()
    rng = np.random.default_rng(2027)
    records = []
    for scenario in ("random", "continuous", "evening"):
        for repetition in range(30):
            mask = missing_mask(len(control), 0.10, scenario, rng)
            incomplete = baseline.copy()
            incomplete[mask] = np.nan
            days = incomplete.reshape(-1, 1440)
            full_days = days[np.isfinite(days).all(axis=1)]
            methods = {
                "available": incomplete,
                "linear_15": interpolate_short(incomplete, 15),
                "linear_60": interpolate_short(incomplete, 60),
                "complete_days": full_days.ravel() if len(full_days) else None,
            }
            for name, values in methods.items():
                metrics = (profile_metrics(values, reference) if values is not None else
                           {"mae_kw": np.nan, "rmse_kw": np.nan,
                            "peak_change_pct": np.nan, "profile_coverage_pct": np.nan})
                records.append({"scenario": scenario, "repetition": repetition + 1,
                                "method": name, **metrics})
    trials = pd.DataFrame(records)
    summary = trials.groupby(["scenario", "method"], as_index=False).agg(
        valid_trials=("mae_kw", "count"), mean_mae_kw=("mae_kw", "mean"),
        mean_rmse_kw=("rmse_kw", "mean"),
        mean_peak_change_pct=("peak_change_pct", "mean"),
        mean_profile_coverage_pct=("profile_coverage_pct", "mean"))
    save_csv(trials, out / "06_h3_trials.csv")
    save_csv(summary, out / "06_h3_summary.csv")
    fig, ax = plt.subplots(figsize=(8, 4))
    order = ["available", "linear_15", "linear_60", "complete_days"]
    for scenario, part in summary.groupby("scenario"):
        part = part.set_index("method").reindex(order)
        ax.plot(order, part.mean_mae_kw, marker="o", label=scenario)
    ax.set(xlabel="Метод", ylabel="Средняя MAE, кВт", title="Сравнение методов")
    ax.legend()
    ax.grid(alpha=0.2)
    fig.tight_layout()
    fig.savefig(out / "06_h3_methods.png", dpi=160)
    plt.close(fig)

## 6.5. Результаты по исходным файлам

Для каждого источника формируется отдельный каталог с таблицами, графиками и метаданными. Проверки, для которых недостаточно полноты или продолжительности наблюдений, не заменяются результатами другого источника.

In [ ]:
report = []
for source in sources:
    name = str(source.relative_to(repo))
    print("\nИсточник:", name)
    try:
        minutes = regular_minutes(load_measurements(source))
    except Exception as exc:
        print("Не удалось прочитать источник:", exc)
        report.append({"source": name, "status": "ошибка чтения", "detail": str(exc)})
        continue

    out = result_folder(source) / "этап_06"
    out.mkdir(parents=True, exist_ok=True)
    coverage = monthly_coverage(minutes)
    save_csv(coverage, out / "06_monthly_coverage.csv")

    # Распределение известных и отсутствующих измерений по месяцам и часам.
    hours = minutes.assign(month=minutes.timestamp.dt.strftime("%Y-%m"),
                           hour=minutes.timestamp.dt.hour)
    natural = hours.groupby(["month", "hour"], as_index=False).agg(
        observed=("Global_active_power", "count"), expected=("timestamp", "size"))
    natural["missing_pct"] = 100 * (1 - natural.observed / natural.expected)
    save_csv(natural, out / "06_natural_missing_by_hour.csv")

    h1 = analyze_h1(minutes, coverage, out)
    control = select_control(minutes)
    if control is None:
        h2 = h3 = "недостаточно 14 полных последовательных суток"
    else:
        reference = analyze_h2(control, out)
        analyze_h3(control, reference, out)
        h2 = h3 = "выполнено"

    save_run_metadata(source, out, {"stage": 6, "h1_periods": h1,
                                    "control_start": str(control.index.min().date()) if control is not None else None,
                                    "control_end": str(control.index.max().date()) if control is not None else None,
                                    "seed_h2": 2026, "seed_h3": 2027})
    report.append({"source": name, "status": "обработан", "h1": h1,
                   "h2": h2, "h3": h3, "result_dir": str(out.relative_to(repo))})
    print("H1:", h1, "| H2:", h2, "| H3:", h3)

summary = pd.DataFrame(report)
save_csv(summary, repo / "результаты/06_сводка_источников.csv")
display(summary)


Источник: data/raw/household_power_consumption.zip


## 6.6. Интерпретация и ограничения

В H1 статистическая значимость оценивается с помощью p-value после поправки Холма при уровне 0,05. Доверительный интервал характеризует неопределённость размера различия при условиях блокового бутстрэпа. Невозможность отвергнуть нулевую гипотезу не доказывает равенство профилей.

В H2–H3 оцениваются абсолютные и относительные искажения при известных исходных значениях. Квантили повторных экспериментов описывают изменчивость рассмотренных сценариев и **не** являются доверительными интервалами по всем домохозяйствам. Причины естественных пропусков и сезонных различий по этим данным не устанавливаются. Для отдельных источников результаты не обобщаются автоматически на другие дома и периоды.